# Derived variables

This notebook shows how to use derived variables. A derived variable is a variable that is not available as an input dataset, but computed from one or more input variables.

In [1]:
import pandas as pd
import yaml

import esmvalcore.preprocessor
from esmvalcore.config import CFG
from esmvalcore.dataset import Dataset, DerivedDataset, datasets_to_recipe

pd.set_option("display.max_colwidth", None)

First, we configure ESMValCore so it searches the ESGF for data:

In [2]:
CFG["projects"]["CMIP6"].pop(
    "data",
    None,
)  # Clear existing CMIP6 configuration for finding input data
CFG.nested_update(
    {
        "projects": {
            "CMIP6": {
                "data": {
                    "intake-esgf": {
                        "type": "esmvalcore.io.intake_esgf.IntakeESGFDataSource",
                        "priority": 2,
                        "facets": {
                            "activity": "activity_drs",
                            "dataset": "source_id",
                            "ensemble": "member_id",
                            "exp": "experiment_id",
                            "institute": "institution_id",
                            "grid": "grid_label",
                            "mip": "table_id",
                            "project": "project",
                            "short_name": "variable_id",
                        },
                    },
                },
            },
        },
    },
)

## Which variables can be derived?

The interface for working with derived variables from Python is not very polished yet. To list all available derived variables, we can run:

In [3]:
pd.DataFrame.from_dict(
    [
        {
            "short_name": short_name,
        }
        | {
            k: getattr(
                esmvalcore.cmor.table.get_tables(
                    CFG,
                    project="CMIP6",
                ).get_variable(
                    table_name="x",
                    short_name=short_name,
                    derived=True,
                ),
                k,
                None,
            )
            for k in ["units", "long_name"]
        }
        for short_name in esmvalcore.preprocessor._derive.ALL_DERIVED_VARIABLES  # noqa: SLF001
    ],
)

,short_name,units,long_name
0,clmmtisccp,%,ISCCP Middle Level Medium-Thickness Cloud Area Fraction
1,sfcwind,NaN,NaN
2,clhtkisccp,%,ISCCP high level thick cloud area fraction
3,rsntcs,W m-2,TOA Net downward Shortwave Radiation assuming clear sky
4,swcre,W m-2,TOA Shortwave Cloud Radiative Effect
5,hfns,W m-2,Surface Net Heat Flux
6,troz,m,Tropospheric Ozone Column (O3 mole fraction < 125 ppb)
7,cllmtisccp,%,ISCCP Low Level Medium-Thickness Cloud Area Fraction
8,lwcre,W m-2,TOA Longwave Cloud Radiative Effect
9,hurs,%,Near-Surface Relative Humidity


Note that [modules, functions, and variables starting with a single `_` character should be considered internal](https://peps.python.org/pep-0008/#descriptive-naming-styles), so there are no guarantees about the stability of this interface.

## Finding available datasets

We define a dataset template to search for all CMIP6 models that provide all required input datasets to derive `lwcre` or longwave cloud radiative effect at the top of atmosphere on a monthly resolution for the historical experiment. Note that ESMValCore uses its own names for the facets for a more uniform naming across different CMIP phases and other projects. The mapping to the facet names used on ESGF can be found in [Facets](https://docs.esmvaltool.org/projects/ESMValCore/en/latest/reference/facets.html).

In [4]:
dataset_template = DerivedDataset(
    short_name="lwcre",
    mip="Amon",
    project="CMIP6",
    exp="historical",
    dataset="*",
    institute="*",
    ensemble="r1i1p1f1",
    grid="gn",
)

Next, we use the `DerivedDataset.from_files` method to build a list of datasets from the available files. This may take a while as searching the ESGF for many files may be a bit slow. Because the search results are cached, subsequent searches will be faster.

In [5]:
datasets = list(dataset_template.from_files())
print(f"Found {len(datasets)} datasets, showing the first 10:")
datasets[:10]

Found 37 datasets, showing the first 10:


[DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=TaiESM1, institute=AS-RCEC, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=AWI-CM-1-1-MR, institute=AWI, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=AWI-ESM-1-1-LR, institute=AWI, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=BCC-CSM2-MR, institute=BCC, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=BCC-ESM1, institute=BCC, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=CAMS-CSM1-0, institute=CAMS, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=CAS-ESM2-0, institute=CAS, ensemble=r1i1p1f1, grid=gn),
 DerivedDataset(sh

## Composing a recipe with derived variables

To use the datasets found above in a recipe, we will want to use the name of the variable that needs to be derived, along with the `derive: true` option:

In [6]:
recipe_datasets = [
    Dataset(
        diagnostic="diagnostic_name",
        derive=True,
        **dataset.facets,
    )
    for dataset in datasets
]
print(yaml.safe_dump(datasets_to_recipe(recipe_datasets)))

datasets:
- dataset: ACCESS-CM2
  institute: CSIRO-ARCCSS
- dataset: ACCESS-ESM1-5
  institute: CSIRO
- dataset: AWI-CM-1-1-MR
  institute: AWI
- dataset: AWI-ESM-1-1-LR
  institute: AWI
- dataset: BCC-CSM2-MR
  institute: BCC
- dataset: BCC-ESM1
  institute: BCC
- dataset: CAMS-CSM1-0
  institute: CAMS
- dataset: CAS-ESM2-0
  institute: CAS
- dataset: CESM2
  institute: NCAR
- dataset: CESM2-FV2
  institute: NCAR
- dataset: CESM2-WACCM
  institute: NCAR
- dataset: CESM2-WACCM-FV2
  institute: NCAR
- dataset: CMCC-CM2-HR4
  institute: CMCC
- dataset: CMCC-CM2-SR5
  institute: CMCC
- dataset: CMCC-ESM2
  institute: CMCC
- dataset: CanESM5
  institute: CCCma
- dataset: CanESM5-1
  institute: CCCma
- dataset: FGOALS-g3
  institute: CAS
- dataset: FIO-ESM-2-0
  institute: FIO-QLNM
- dataset: GISS-E2-1-G
  institute: NASA-GISS
- dataset: GISS-E2-1-G-CC
  institute: NASA-GISS
- dataset: GISS-E2-1-H
  institute: NASA-GISS
- dataset: GISS-E2-2-G
  institute: NASA-GISS
- dataset: GISS-E2-2-H
  

There is also a `force_derivation` option available for use in the recipe, when set to `true` that will cause the variable to be derived even if it is already available as a dataset.

## Computing the derived variable

Let's load the data to derive the first dataset:

In [7]:
dataset = datasets[0]
dataset

DerivedDataset(short_name=lwcre, mip=Amon, project=CMIP6, exp=historical, dataset=TaiESM1, institute=AS-RCEC, ensemble=r1i1p1f1, grid=gn)

In [8]:
cubes = dataset.load()
cubes

 rlut: attribute positive not present
loaded from file 
 rlutcs: attribute positive not present
loaded from file 


<iris 'Cube' of TOA Longwave Cloud Radiative Effect / (W m-2) (time: 1980; latitude: 192; longitude: 288)>

## Implementing your own derived variables

Guidance on adding new built-in derived variables to ESMValCore is available in [Deriving a variable](https://docs.esmvaltool.org/projects/ESMValCore/en/latest/develop/derivation.html). However, if you are only using the Python interface, you can define an ad-hoc derived variable by subclassing the `DerivedDataset` class and implementing a custom `required` attribute and `derive` method. The `required` attribute defines the facets that describe the input data:

In [9]:
dataset.required

[{'short_name': 'rlut'}, {'short_name': 'rlutcs'}]

in this case we see that `lwcre` is derived from variables `rlut` and `rlutcs`. The `derive` method is a function that takes the iris cubes resulting from loading the datasets described by the `facets` and `required` attribute as an argument, and computes the derived variable.